# Football Match Prediction: XGBoost Model
Notebook này sử dụng tập dữ liệu One-Big-Table (`feature_match_ml.parquet`) từ lớp Gold để huấn luyện mô hình học máy phân loại (Classification) dự đoán kết quả trận đấu (Home, Draw, Away).

In [ ]:
import sys
import os
from pathlib import Path

# Đảm bảo import được module `lake`
try:
    from lake import minio_io as mio
except ImportError:
    # Nếu chạy trên Colab / local root
    cwd = Path.cwd()
    root = cwd.parent.parent.parent if cwd.name == 'ML' else cwd
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    from lake import minio_io as mio

# Sửa lỗi Invalid endpoint cho MinIO nếu thiếu http://
if os.environ.get("MINIO_ENDPOINT", "").startswith("20.41"):
    os.environ["MINIO_ENDPOINT"] = "http://" + os.environ["MINIO_ENDPOINT"]

import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb

pd.set_option('display.max_columns', 100)
sns.set_theme(style="whitegrid")

In [ ]:
# Lấy danh sách file parquet trong thư mục gold/features
print("Đang tải dữ liệu OBT từ MinIO...")
keys = list(mio.list_keys("gold/features/feature_match_ml"))
if not keys:
    print("❌ Không tìm thấy feature_match_ml trong thư mục gold/features. Bạn đã chạy script build_features_v5.py chưa?")
else:
    # Lấy file mới nhất
    latest_key = sorted(keys, key=lambda x: x[0])[-1][0]
    print(f"File OBT: {latest_key}")
    
    # Đọc parquet trực tiếp từ RAM
    df = pd.read_parquet(io.BytesIO(mio.read_bytes(latest_key)))
    
    # Chỉ giữ các trận đấu 'warm' (các đội đã đá đủ số trận tối thiểu để có rolling features)
    df = df[df['is_warm'] == 1].copy()
    print(f"✅ Tải thành công {len(df)} trận đấu (is_warm=1)!")

## Tiền xử lý dữ liệu (Data Preprocessing)
XGBoost có thể xử lý giá trị bị thiếu (NaN) tự động, nhưng ta cần chuyển đổi cột Target ('H', 'D', 'A') thành số nguyên.

In [ ]:
# Xác định các cột không dùng làm Feature ML
ignore_cols = [
    'match_id', 'division', 'season', 'match_date', 'home_key', 'away_key', 
    'home_team', 'away_team', 'split', 'is_warm',
    'target', 'target_home_goals', 'target_away_goals', 'target_total_goals', 
    'target_over25', 'target_btts'
]

# Các cột feature thực sự
feature_cols = [col for col in df.columns if col not in ignore_cols]
print(f"Số lượng Features: {len(feature_cols)}")

# Encode Target: H (Home) -> 2, D (Draw) -> 1, A (Away) -> 0
# Hoặc mã hóa bằng LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df['target'])
print("Target mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# Data phân tách train/val/test theo logic của cột 'split' hoặc thời gian
# Ta sẽ sort theo thời gian và chia train/test theo mốc thời gian để tránh Data Leakage
df = df.sort_values('match_date')

# Tập Train (Từ đầu đến cuối năm 2023)
# Tập Test (Năm 2024 trở đi)
train_mask = df['match_date'] < '2024-01-01'
test_mask = df['match_date'] >= '2024-01-01'

X_train = df.loc[train_mask, feature_cols]
y_train = y[train_mask]

X_test = df.loc[test_mask, feature_cols]
y_test = y[test_mask]

print(f"Tập Train: {X_train.shape[0]} trận")
print(f"Tập Test: {X_test.shape[0]} trận")

## Huấn luyện Mô hình XGBoost (Training)

In [ ]:
# Khởi tạo mô hình
# Các tham số cơ bản chống Overfitting
model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='multi:softprob',
    eval_metric='mlogloss'
)

# Huấn luyện mô hình
print("Đang huấn luyện mô hình XGBoost...")
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=20
)
print("✅ Huấn luyện hoàn tất!")

## Đánh giá Mô hình (Evaluation)

In [ ]:
# Dự đoán trên tập Test
y_pred = model.predict(X_test)
y_pred_probs = model.predict_proba(X_test)

print("ĐỘ CHÍNH XÁC (ACCURACY):", round(accuracy_score(y_test, y_pred), 4))
print("\n--- BÁO CÁO PHÂN LOẠI (CLASSIFICATION REPORT) ---")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Vẽ Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Ma trận Nhầm lẫn (Confusion Matrix)')
plt.ylabel('Thực tế (Actual)')
plt.xlabel('Dự đoán (Predicted)')
plt.show()

## Tầm quan trọng của Features (Feature Importance)
Cùng xem XGBoost đánh giá các Feature mới như Quãng đường (travel_distance), Thời tiết, Độ ồn truyền thông, xG... có thực sự tác động mạnh không.

In [ ]:
# Lấy top 30 Feature quan trọng nhất
importances = model.feature_importances_
indices = np.argsort(importances)[::-1][:30]
top_features = [feature_cols[i] for i in indices]
top_importances = importances[indices]

plt.figure(figsize=(12, 10))
sns.barplot(x=top_importances, y=top_features, palette='viridis')
plt.title('Top 30 Feature Quan Trọng Nhất (XGBoost Feature Importance)')
plt.xlabel('Mức độ đóng góp')
plt.ylabel('Tên Feature')
plt.show()